# Mini-Courier Planner: CSV Loader and Algorithm Demonstration

This notebook uses the project modules directly. Run cells from the project root with the selected Python 3.13.14 environment. The route file is an undirected weighted graph; Nearest Neighbor is a TSP approximation, not a guarantee of the optimal tour.

## 1. Inspect and Fix CSV Loader and Data Files

The CSVs are the source of truth for the application. `routes.csv` must begin with `source,destination,distance` and contain the eight specified road edges. The loader uses `pathlib`-resolved defaults, skips blank records, validates required columns and row shape, and raises path- and row-specific errors for invalid input.

In [1]:
from pathlib import Path
import csv

ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "datasets" / "routes.csv").exists()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from within the Mini-Courier-Planner project.")

for relative_path in ("datasets/routes.csv", "datasets/parcels.csv"):
    path = ROOT / relative_path
    print(f"--- {relative_path} ---")
    print(path.read_text(encoding="utf-8-sig"))

--- datasets/routes.csv ---
source,destination,distance
Warehouse,A,4
Warehouse,B,2
Warehouse,D,12
A,B,1
A,C,5
B,C,8
B,D,10
C,D,2

--- datasets/parcels.csv ---
parcel_id,priority,weight,value,deadline
P101,2,2,30,2026-10-22
P102,5,3,40,2026-10-21
P103,1,4,50,2026-10-24
P104,4,5,60,2026-10-22
P105,3,1,20,2026-10-23
P106,5,2,45,2026-10-21
P107,2,3,25,2026-10-25
P108,4,2,35,2026-10-22



## 2. Verify Data Loading Functionality

The following cell invokes the same module entry point as `python -m src.data_loader`, then loads the records into Python structures and verifies the expected 8 parcels and 5 unique locations.

In [2]:
import runpy
from src.data_loader import load_graph, load_parcels

runpy.run_module("src.data_loader", run_name="__main__")
parcels = load_parcels(ROOT / "datasets" / "parcels.csv")
graph = load_graph(ROOT / "datasets" / "routes.csv")
print("Parcel count:", len(parcels))
print("Location count:", len(graph))
print("Parcels:")
for parcel in parcels:
    print(parcel)
print("Undirected routes:")
for source, neighbors in graph.items():
    print(source, neighbors)
assert len(parcels) == 8
assert len(graph) == 5

Parcels loaded: 8
Locations loaded: 5
First parcel: {'id': 'P101', 'priority': 2, 'weight': 2, 'value': 30, 'deadline': '2026-10-22'}
Warehouse connections: {'A': 4.0, 'B': 2.0, 'D': 12.0}
Parcel count: 8
Location count: 5
Parcels:
{'id': 'P101', 'priority': 2, 'weight': 2, 'value': 30, 'deadline': '2026-10-22'}
{'id': 'P102', 'priority': 5, 'weight': 3, 'value': 40, 'deadline': '2026-10-21'}
{'id': 'P103', 'priority': 1, 'weight': 4, 'value': 50, 'deadline': '2026-10-24'}
{'id': 'P104', 'priority': 4, 'weight': 5, 'value': 60, 'deadline': '2026-10-22'}
{'id': 'P105', 'priority': 3, 'weight': 1, 'value': 20, 'deadline': '2026-10-23'}
{'id': 'P106', 'priority': 5, 'weight': 2, 'value': 45, 'deadline': '2026-10-21'}
{'id': 'P107', 'priority': 2, 'weight': 3, 'value': 25, 'deadline': '2026-10-25'}
{'id': 'P108', 'priority': 4, 'weight': 2, 'value': 35, 'deadline': '2026-10-22'}
Undirected routes:
Warehouse {'A': 4.0, 'B': 2.0, 'D': 12.0}
A {'Warehouse': 4.0, 'B': 1.0, 'C': 5.0}
B {'Wareho

<frozen runpy>:130: RuntimeWarning: 'src.data_loader' found in sys.modules after import of package 'src', but prior to execution of 'src.data_loader'; this may result in unpredictable behaviour


## 3. Update and Validate Unit Tests for Data Loading

`tests/test_data_loader.py` exercises successful CSV loading, undirected route construction, missing files, invalid columns, malformed numeric fields, malformed headers, and non-positive parcel weights. Run this cell from the notebook to execute the suite and see the test summary.

In [3]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
    cwd=ROOT,
    text=True,
    capture_output=True,
)
print(result.stdout)
print(result.stderr)
if result.returncode:
    raise RuntimeError(f"unittest exited with status {result.returncode}")


test_invalid_columns_have_clear_error (test_data_loader.DataLoaderTests.test_invalid_columns_have_clear_error) ... ok
test_loads_project_datasets (test_data_loader.DataLoaderTests.test_loads_project_datasets) ... ok
test_malformed_header_and_route_number (test_data_loader.DataLoaderTests.test_malformed_header_and_route_number) ... ok
test_malformed_values_and_non_positive_weight (test_data_loader.DataLoaderTests.test_malformed_values_and_non_positive_weight) ... ok
test_missing_file_has_clear_error (test_data_loader.DataLoaderTests.test_missing_file_has_clear_error) ... ok
test_distance_and_reconstructed_path (test_project.DijkstraTests.test_distance_and_reconstructed_path) ... ok
test_unreachable_destination (test_project.DijkstraTests.test_unreachable_destination) ... ok
test_invalid_capacity_and_weight (test_project.KnapsackTests.test_invalid_capacity_and_weight) ... ok
test_optimal_value_and_capacity (test_project.KnapsackTests.test_optimal_value_and_capacity) ... ok
test_merge_so

## 4. Demonstrate Data Loading in Notebook

Display the parsed records as compact Markdown tables without introducing a separate CSV implementation or extra data-processing dependency.

In [4]:
from IPython.display import Markdown, display

parcel_columns = ["id", "priority", "weight", "value", "deadline"]
parcel_table = "| " + " | ".join(parcel_columns) + " |\n"
parcel_table += "| " + " | ".join(["---"] * len(parcel_columns)) + " |\n"
for parcel in parcels:
    parcel_table += "| " + " | ".join(str(parcel[column]) for column in parcel_columns) + " |\n"
display(Markdown("### Parcels\n" + parcel_table))

route_table = "| source | destination | distance (km) |\n| --- | --- | ---: |\n"
seen_edges = set()
for source, neighbors in graph.items():
    for destination, distance in neighbors.items():
        edge = frozenset((source, destination))
        if edge not in seen_edges:
            route_table += f"| {source} | {destination} | {distance:g} |\n"
            seen_edges.add(edge)
display(Markdown("### Routes (undirected edges shown once)\n" + route_table))

### Parcels
| id | priority | weight | value | deadline |
| --- | --- | --- | --- | --- |
| P101 | 2 | 2 | 30 | 2026-10-22 |
| P102 | 5 | 3 | 40 | 2026-10-21 |
| P103 | 1 | 4 | 50 | 2026-10-24 |
| P104 | 4 | 5 | 60 | 2026-10-22 |
| P105 | 3 | 1 | 20 | 2026-10-23 |
| P106 | 5 | 2 | 45 | 2026-10-21 |
| P107 | 2 | 3 | 25 | 2026-10-25 |
| P108 | 4 | 2 | 35 | 2026-10-22 |


### Routes (undirected edges shown once)
| source | destination | distance (km) |
| --- | --- | ---: |
| Warehouse | A | 4 |
| Warehouse | B | 2 |
| Warehouse | D | 12 |
| A | B | 1 |
| A | C | 5 |
| B | C | 8 |
| B | D | 10 |
| D | C | 2 |


## Algorithm Demonstration

Priority controls the Merge Sort order; deadlines are visible dispatch metadata but do not modify ordering. Knapsack maximizes parcel value under the 8 kg capacity. Dijkstra finds the shortest route from Warehouse to D. The Nearest Neighbor route visits each location and returns to Warehouse when possible; it is a heuristic approximation and may not be optimal.

In [5]:
from src.dijkstra import dijkstra, get_path
from src.knapsack import knapsack
from src.sorting import merge_sort
from src.tsp import nearest_neighbor

sorted_parcels = merge_sort(parcels)
selected, total_value = knapsack(parcels, capacity=8)
distances, previous = dijkstra(graph, "Warehouse")
shortest_path = get_path(previous, "D", "Warehouse")
tour, tour_distance = nearest_neighbor(graph, "Warehouse")

print("Sorted parcel IDs:", [parcel["id"] for parcel in sorted_parcels])
print("Selected parcels:", [parcel["id"] for parcel in selected])
print("Packed weight:", sum(parcel["weight"] for parcel in selected), "/ 8 kg")
print("Total parcel value:", total_value)
print("Shortest route to D:", " -> ".join(shortest_path), f"({distances['D']:g} km)")
print("Approximate delivery tour:", " -> ".join(tour), f"({tour_distance:g} km)")

Sorted parcel IDs: ['P102', 'P106', 'P104', 'P108', 'P105', 'P101', 'P107', 'P103']
Selected parcels: ['P102', 'P105', 'P106', 'P108']
Packed weight: 8 / 8 kg
Total parcel value: 140
Shortest route to D: Warehouse -> B -> A -> C -> D (10 km)
Approximate delivery tour: Warehouse -> B -> A -> C -> D -> Warehouse (22 km)


## Performance Analysis

The benchmark module uses deterministic synthetic inputs seeded with 42, tests sizes 10, 50, 100, and 200, holds knapsack capacity at 50, and reports the median of five `time.perf_counter()` measurements. Timings are machine-dependent and especially noisy at these small durations. Complexity summary: Merge Sort $O(n log n)$ time/$O(n)$ space; Knapsack DP $O(nW)$ time and space; heap-based Dijkstra $O((V+E) log V)$ time/$O(V+E)$ space; Nearest Neighbor TSP $O(V^2)$ time/$O(V)$ space.

In [6]:
from src.performance import run_benchmarks

benchmark_results = run_benchmarks(output_dir=ROOT / "images")
benchmark_results


RUNTIME PERFORMANCE ANALYSIS
Median of 5 runs; seconds
------------------------------------------------------------------------------
Input size    Merge Sort        Knapsack DP       Dijkstra          
10            0.000011          0.000060          0.000015          
50            0.000050          0.000267          0.000103          
100           0.000091          0.000430          0.000214          
200           0.000175          0.000787          0.000689          



Results saved to C:\Users\Yesh Verma\Mini-Courier-Planner\images\performance_results.csv
Chart saved to C:\Users\Yesh Verma\Mini-Courier-Planner\images\runtime_chart.png


[{'input_size': 10,
  'merge_sort_seconds': 1.0599993402138352e-05,
  'knapsack_dp_seconds': 5.990000499878079e-05,
  'dijkstra_seconds': 1.4999997802078724e-05},
 {'input_size': 50,
  'merge_sort_seconds': 5.03999981447123e-05,
  'knapsack_dp_seconds': 0.0002669000023161061,
  'dijkstra_seconds': 0.00010290000500390306},
 {'input_size': 100,
  'merge_sort_seconds': 9.139999747276306e-05,
  'knapsack_dp_seconds': 0.0004303999958210625,
  'dijkstra_seconds': 0.00021410000044852495},
 {'input_size': 200,
  'merge_sort_seconds': 0.00017500000103609636,
  'knapsack_dp_seconds': 0.0007868999964557588,
  'dijkstra_seconds': 0.0006894000034662895}]